In [5]:
import pandas as pd

In [4]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")


In [4]:
print("Training data shape:", train_df.shape)
print("Testing data shape:", test_df.shape)
print("\nTraining columns:")
print(train_df.columns.tolist())
print("\nTesting columns:")
print(test_df.columns.tolist())



Training data shape: (159571, 8)
Testing data shape: (153164, 2)

Training columns:
['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

Testing columns:
['id', 'comment_text']


In [5]:
train_df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [6]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   id             159571 non-null  str  
 1   comment_text   159571 non-null  str  
 2   toxic          159571 non-null  int64
 3   severe_toxic   159571 non-null  int64
 4   obscene        159571 non-null  int64
 5   threat         159571 non-null  int64
 6   insult         159571 non-null  int64
 7   identity_hate  159571 non-null  int64
dtypes: int64(6), str(2)
memory usage: 72.2 MB


In [7]:
train_df.describe()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
count,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000
mean,0.095844,0.009996,0.052948,0.002996,0.049364,0.008805
std,0.294379,0.099477,0.223931,0.054650,0.216627,0.093420
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


##### Separate X and y


In [6]:
target_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

train_df[target_columns].sum()

toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64

In [9]:
train_df[target_columns].mean() * 100

toxic            9.584448
severe_toxic     0.999555
obscene          5.294822
threat           0.299553
insult           4.936361
identity_hate    0.880486
dtype: float64

In [7]:
X = train_df["comment_text"]
y = train_df[target_columns]

In [8]:
print(X.shape)
print(y.shape)

(159571,)
(159571, 6)


##### Clean the Text

In [9]:
import re

In [10]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [11]:
X_clean = X.apply(clean_text)

In [12]:
print(X.iloc[1])
print("\n")
print(X_clean.iloc[1])

D'aww! He matches this background colour I'm seemingly stuck with. Thanks.  (talk) 21:51, January 11, 2016 (UTC)


daww he matches this background colour im seemingly stuck with thanks talk january utc


#### Tokenization

In [1]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [14]:
tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_clean)

In [ ]:
X_sequences = tokenizer.texts_to_sequences(X_clean)

In [16]:
print(X_sequences[0])

[639, 76, 2, 123, 127, 174, 29, 629, 4522, 11331, 1041, 83, 312, 53, 2011, 10779, 51, 6445, 16, 62, 2606, 144, 8, 2760, 34, 115, 1132, 15137, 2793, 5, 46, 55, 235, 2, 410, 31, 2, 42, 28, 142, 70, 3338, 90]


In [18]:
len(X_sequences)

159571

##### Padding

In [19]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [20]:
X_padded = pad_sequences(
    X_sequences,
    maxlen=200,
    padding="post",
    truncating="post"
)

In [21]:
print(X_padded.shape)

(159571, 200)
